> **교육 목표** [팀 워크숍] 제철소 전력 사용량(`usage_kwh`)을 예측하고, 사용량을 좌우하는 조건을 설명합니다.
>
> **핵심 내용** 날짜 변환·EDA → 누수 점검 → 피처 → 시간 기준 분할 → 평균 기준선 → 선형 vs ML → 오차 분석 → 중요 변수 (성능 40 · 과정 30 · 해석 30)

In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D4_W 팀 워크숍: 제철소 전력 사용량 예측

> **미션:** 전력 사용량(`usage_kwh`)을 예측하고, 어떤 조건이 사용량을 좌우하는지 설명하기

| 항목 | 내용 |
|---|---|
| 데이터 | 국내 제철소 2018년, 15분 간격 35,040건 |
| 평가 데이터 | **시간 순 마지막 20%** (약 10월 하순 ~ 12월) |
| 평가 기준 | 성능 40% (Test MAE, kWh) · 과정 30% (누수 점검, 시간 분할, 기준선 비교) · 해석 30% (중요 변수, 오차 큰 구간) |
| 시간 | 6교시: 1\~5단계 / 7교시: 6\~8단계 + 3분 공유 |

| 컬럼 | 뜻 |
|---|---|
| `date` | 측정 일시 (일/월/년 시:분) |
| `usage_kwh` | 전력 사용량 ← 예측 대상 |
| `lag_kvarh`, `lead_kvarh` | 지상·진상 무효전력 |
| `co2` | 이산화탄소 배출량 |
| `lag_pf`, `lead_pf` | 지상·진상 역률 |
| `nsm` | 자정 이후 경과 초 |
| `week_status`, `day_of_week` | 주중/주말, 요일 |
| `load_type` | 부하 유형 (Light / Medium / Maximum) |

<div style="clear: both; height: 20px;"></div>

> ⚠️ 이 데이터에는 **정답을 거의 그대로 담은 컬럼**이 숨어 있습니다. 2단계에서 반드시 찾아서 빼세요.

## 0. 준비 📋

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

se = pd.read_csv("../../data/steel_energy.csv")

## 1. EDA: 날짜 변환과 패턴 ✍️

날짜를 변환하고 **시간 순으로 정렬됐는지** 확인합니다. `False`가 나오면 원본 순서가 시간 순이 아닌 곳이 있다는 뜻이므로 정렬한 뒤 씁니다.
(이 원본은 매일 00:00 기록이 그날의 맨 끝에 붙어 있습니다.) 시간대별·요일별 평균 사용량도 봅니다.

In [ ]:
# 힌트: pd.to_datetime(컬럼, format="%d/%m/%Y %H:%M"), 표.sort_values("컬럼")

## 2. 누수 점검 ✍️

예측 대상과 상관이 너무 높은 컬럼을 찾습니다. **예측하는 시점에 그 값을 알 수 있는지**도 함께 따져 보세요.

In [ ]:
# 힌트: 표.corr(numeric_only=True)["대상"]

CO2를 빼도 정말 괜찮을까요? 무효전력·역률을 넣은 모델을 시험해 봅니다. 📋

In [ ]:
cut = int(len(se) * 0.8)
test_cols = ["lag_kvarh", "lead_kvarh", "lag_pf", "lead_pf", "nsm"]     # CO2만 뺀 측정값들
rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(se[test_cols].iloc[:cut], se["usage_kwh"].iloc[:cut])
pred = rf.predict(se[test_cols].iloc[cut:])
print(f"CO2 없이 무효전력·역률로: R² {r2_score(se['usage_kwh'].iloc[cut:], pred):.3f}")

**Q1.** 어떤 컬럼을 입력에서 빼야 할까요? 이유는?

> 답:


## 3. 입력 피처 만들기 ✍️

시각(`hour`)과 범주 컬럼을 0/1 컬럼으로 바꿉니다(`pd.get_dummies`).

> 변수 이름: `X`, `y`

In [ ]:
# 힌트: pd.get_dummies(표[범주컬럼들]).astype(int), pd.concat([표1, 표2], axis=1)

## 4. 시간 기준 분할 ✍️

## 5. 평균 기준선 ✍️

In [ ]:
# 힌트: DummyRegressor()

## 6. 선형 vs ML ✍️

## 7. 오차 분석 ✍️

어느 시간대·요일에 크게 틀리는지 봅니다. 하루치 예측-실제 그래프도 그려 보세요.

In [ ]:
# 힌트: 표.groupby("컬럼")["오차"].mean()

## 8. 중요 변수 ✍️

## 중간 점검 체크리스트

- [ ] 시간 기준으로 나눴는가? (앞 80% Train / 마지막 20% Test)
- [ ] CO2 등 정답을 담은 컬럼(동시 측정값)을 입력에서 뺐는가?
- [ ] 평균 예측 기준선과 비교했는가?
- [ ] 전처리 기준을 Train에서만 계산했는가?
- [ ] MAE를 kWh 단위로 해석했는가?

## 결과 공유 (팀당 3분) 💬

**1. 문제와 데이터 (30초)** 무엇을 예측했고, 무엇을 뺐나?

> 답:


**2. 접근 방법 (1분)** 분할, 기준선, 비교한 모델

> 답:


**3. 결과 (1분)** MAE, 예측-실제 그래프, 중요 변수

> 답:


**4. 배운 점 (30초)** 다음에 다르게 해 볼 것

> 답:
